# KAN-Vim: toàn bộ thí nghiệm trên Colab (GPU trả phí)

**Trước khi chạy**
1. Upload `kan-vim-v3.zip` vào **MyDrive** (thư mục gốc Google Drive).
2. **Runtime → Change runtime type → A100 GPU** (hoặc L4) → Save. Nếu có, bật **High-RAM**.

**Cách chạy:** Cell 1 → Cell 2. Để tab mở.
- Kết quả lưu thẳng lên Drive: `MyDrive/kan-vim-results/results_plan`.
- **Bị ngắt / hết compute units / hết giờ:** mở lại notebook, chạy lại Cell 1 → Cell 2. Script tự bỏ qua phần đã xong và chạy tiếp lần train đang dở từ epoch cuối.
- Cell 3: xem báo cáo bất cứ lúc nào (kể cả khi đang chạy dở, chạy ở một tab khác hoặc sau khi dừng Cell 2).


In [ ]:
# 1. Chuẩn bị (mỗi lần mở notebook đều chạy lại)
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, zipfile, shutil
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
ZIP = '/content/drive/MyDrive/kan-vim-v3.zip'
assert os.path.exists(ZIP), 'Không thấy MyDrive/kan-vim-v3.zip: upload lên Drive rồi chạy lại'
shutil.rmtree('/content/kan-vim', ignore_errors=True)
zipfile.ZipFile(ZIP).extractall('/content')
OUT = '/content/drive/MyDrive/kan-vim-results/results_plan'
os.makedirs(os.path.dirname(OUT), exist_ok=True)
!pip -q install huggingface_hub matplotlib
import torch; print('cuda ok:', torch.cuda.is_available(), '| CPU:', os.cpu_count())
print('đã có:', sorted(os.listdir(OUT)) if os.path.exists(OUT) else 'chưa có gì (lần đầu)')


In [ ]:
# 2. Chạy toàn bộ 9 bước (tự chạy tiếp nếu trước đó bị ngắt). Đổi --budget_hours theo số giờ bạn có.
!cd /content/kan-vim && python scripts/run_plan.py --profile colab --budget_hours 30     --out /content/drive/MyDrive/kan-vim-results/results_plan --data_root /content/data --ckpt /content/vim_t.pth


In [ ]:
# 3. Xem báo cáo (chạy được cả khi mới xong một phần)
import os, subprocess
from IPython.display import Markdown, Image, display
R = '/content/drive/MyDrive/kan-vim-results/results_plan'
subprocess.run(['python', '/content/kan-vim/scripts/report_plan.py', R])
if os.path.exists(f'{R}/BAO_CAO.md'):
    display(Markdown(open(f'{R}/BAO_CAO.md', encoding='utf-8').read()))
    for f in sorted(os.listdir(R)):
        if f.startswith('fig_'):
            display(Image(f'{R}/{f}'))
else:
    print('Chưa có báo cáo')
